In [1]:
import numpy as np

# Let's pretend we have 500 customers
num_customers = 500

print("--- Step 1: Generating Data ---")

# 1. Sequence Data: Spend history over the last 3 months (e.g., [$50, $60, $55])
# Shape needed for LSTM: (Samples, Time Steps, Features) -> (500, 3, 1)
raw_sequence_data = np.random.uniform(10, 200, size=(num_customers, 3))
X_seq = np.expand_dims(raw_sequence_data, axis=-1)

# 2. Static Data: Customer Age and Loyalty Tier (0=Bronze, 1=Silver, 2=Gold)
# Shape needed for Dense: (Samples, Features) -> (500, 2)
X_static = np.random.uniform(18, 70, size=(num_customers, 2))
# Let's clean up the tiers to be integers (0, 1, or 2)
X_static[:, 1] = np.random.randint(0, 3, size=(num_customers,))

# 3. Target Data: Actual spend next month (what we want to predict)
y_target = (
    (X_seq[:, -1, 0] * 1.1)
    + (X_static[:, 1] * 20)
    + np.random.normal(0, 5, size=(num_customers, 1))
)

print(f"Sequence Input Shape (LSTM): {X_seq.shape}")
print(f"Static Input Shape (Dense):  {X_static.shape}")
print(f"Target Output Shape:         {y_target.shape}")


--- Step 1: Generating Data ---
Sequence Input Shape (LSTM): (500, 3, 1)
Static Input Shape (Dense):  (500, 2)
Target Output Shape:         (500, 500)


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model

print("\n--- Step 2: Building Hybrid Model ---")

# --- BRANCH 1: LSTM for Sequence Data ---
seq_input = layers.Input(shape=(3, 1), name="sequence_input")
lstm_branch = layers.LSTM(32, return_sequences=False)(seq_input)
lstm_branch = layers.Dense(16, activation="relu")(lstm_branch)

# --- BRANCH 2: Dense MLP for Static Metadata ---
static_input = layers.Input(shape=(2,), name="static_input")
dense_branch = layers.Dense(32, activation="relu")(static_input)
dense_branch = layers.Dense(16, activation="relu")(dense_branch)

# --- THE MERGE LAYER ---
# Combines the features extracted from both branches into one vector
merged_vectors = layers.concatenate([lstm_branch, dense_branch])

# --- FINAL HEAD: Making the regression prediction ---
final_dense = layers.Dense(16, activation="relu")(merged_vectors)
output_layer = layers.Dense(1, name="output")(final_dense)

# Define the unified model entry and exit points
model = Model(inputs=[seq_input, static_input], outputs=output_layer)

# Compile
model.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])

print("Architecture successfully wired. Starting training...")

# Train using a list of our two separate input arrays
model.fit([X_seq, X_static], y_target, epochs=15, batch_size=16, validation_split=0.2)



--- Step 2: Building Hybrid Model ---
Architecture successfully wired. Starting training...
Epoch 1/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 20040.7422 - mae: 127.0843 - val_loss: 18762.5762 - val_mae: 121.9654
Epoch 2/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 17167.6504 - mae: 115.1904 - val_loss: 15021.3184 - val_mae: 105.7697
Epoch 3/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 12799.4541 - mae: 95.5461 - val_loss: 9866.2139 - val_mae: 81.8140
Epoch 4/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 7778.5952 - mae: 71.9024 - val_loss: 5703.8101 - val_mae: 62.0167
Epoch 5/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 5213.7285 - mae: 59.7087 - val_loss: 5175.2026 - val_mae: 59.6079
Epoch 6/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4960.2339 - mae: 58.5323 - val_loss: 5071.8364 - val_mae: 59.0946
Epoch 7/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4890.3296 - mae: 58.1735 - val_loss: 4977.1143 - val_mae: 58.6211
Epoch 8/15
25/25 ━━━━━━━━━━━

In [3]:
print("\n--- Step 3: Production Deployment Simulation ---")

# 1. Save the model to disk
model.save("ecommerce_hybrid_model.keras")
print("Model saved safely as 'ecommerce_hybrid_model.keras'")

# 2. Clear memory and load it back up (mimicking a server restart)
production_model = tf.keras.models.load_model("ecommerce_hybrid_model.keras")



--- Step 3: Production Deployment Simulation ---
Model saved safely as 'ecommerce_hybrid_model.keras'


In [4]:
# 3. Incoming Live Customer Data Payload from a web API:
# This customer spent $120, $140, and $150 over the last 3 months.
# They are 35 years old and in Loyalty Tier 2 (Gold).
live_history = [120.0, 140.0, 150.0]
live_metadata = [35.0, 2.0]

# 4. CRITICAL STEP: Shape them to match the exact training matrix layout
live_seq_3d = np.array(live_history).reshape(1, 3, 1)  # (1 Sample, 3 Steps, 1 Feature)
live_static_2d = np.array([live_metadata])  # (1 Sample, 2 Features)

# 5. Run Inference
prediction = production_model.predict([live_seq_3d, live_static_2d], verbose=0)

print(
    f"\n[API Response] Predicted spend for this customer next month: ${prediction[0][0]:.2f}"
)



[API Response] Predicted spend for this customer next month: $121.04


In [5]:
# 1. Load your deployed production model
production_model = tf.keras.models.load_model("ecommerce_hybrid_model.keras")

# 2. Incoming Live Customer Data Payload:
# This customer spent very little: $15, $12, and $20 over the last 3 months.
# They are 21 years old and in Loyalty Tier 0 (Bronze).
new_live_history = [15.0, 12.0, 20.0]
new_live_metadata = [21.0, 0.0]

# 3. Shape them to match the 3D and 2D matrix layouts
new_seq_3d = np.array(new_live_history).reshape(
    1, 3, 1
)  # (1 Sample, 3 Steps, 1 Feature)
new_static_2d = np.array([new_live_metadata])  # (1 Sample, 2 Features)

# 4. Run Inference
new_prediction = production_model.predict([new_seq_3d, new_static_2d], verbose=0)

print(
    f"\n[API Response] Predicted spend for this customer next month: ${new_prediction[0][0]:.2f}"
)


[API Response] Predicted spend for this customer next month: $104.69
